# Herencia, composición y clases abstractas (ABC)

Hasta ahora cada clase vivía por su cuenta. Pero en programación orientada a objetos
las clases se **relacionan** entre sí, y elegir bien el tipo de relación es una de las
decisiones de diseño más importantes.

En este vídeo veremos tres herramientas, en tres partes:

- **Herencia**: reutilizar comportamiento con una jerarquía "**es un**" (*is-a*).
- **Composición**: construir objetos a partir de otros, una relación "**tiene un**"
  (*has-a*).
- **Clases abstractas (ABC)**: definir un **contrato** que las subclases están
  obligadas a cumplir.

> 📚 Documentación oficial de referencia:
> **[Herencia](https://docs.python.org/es/3/tutorial/classes.html#inheritance)** y
> **[abc](https://docs.python.org/es/3/library/abc.html)**.

---

# Parte 1: Herencia

## 1. Reutilizar comportamiento con una jerarquía

La **herencia** permite que una clase (la **subclase** o clase hija) parta de otra
(la **clase base** o clase padre), heredando sus atributos y métodos. Se usa cuando
existe una relación "**es un**": un perro **es un** animal, un gato **es un** animal.

La subclase puede **sobrescribir** (redefinir) métodos de la base para darles su
propio comportamiento:

In [ ]:
class Animal:
    def __init__(self, nombre):
        self.nombre = nombre

    def hablar(self):
        return "..."


class Perro(Animal):          # Perro hereda de Animal
    def hablar(self):         # sobrescribe el método de la base
        return "Guau"


class Gato(Animal):
    def hablar(self):
        return "Miau"


for animal in [Perro("Toby"), Gato("Misi")]:
    # nombre viene de Animal; hablar() es el de cada subclase
    print(animal.nombre, "dice", animal.hablar())

Fíjate en que `Perro` y `Gato` **no** redefinen `__init__`: lo heredan de `Animal`,
así que `self.nombre` funciona sin escribir nada. Solo cambian lo que necesitan
(`hablar`).

## 2. `super()`: ampliar sin repetir

¿Y si una subclase necesita su **propio estado** además del de la base? Podríamos
reescribir el `__init__` entero, pero eso duplica código. En su lugar usamos
`super()`, que nos da acceso a la clase base para reutilizar su `__init__`:

In [ ]:
class Animal:
    def __init__(self, nombre):
        self.nombre = nombre

    def describir(self):
        return f"{self.nombre} es un animal"


class Perro(Animal):
    def __init__(self, nombre, raza):
        super().__init__(nombre)  # reutiliza el __init__ de Animal (asigna nombre)
        self.raza = raza          # y añade su propio estado

    def describir(self):
        return f"{self.nombre} es un perro de raza {self.raza}"


toby = Perro("Toby", "labrador")
print(toby.nombre, "-", toby.raza)
print(toby.describir())

> 💡 `super().__init__(...)` llama al constructor de la clase base. Es la forma
> correcta de **ampliar** el comportamiento del padre en vez de reemplazarlo: dejas
> que la base haga su parte y tú añades la tuya.

---

# Parte 2: Composición

## 3. "Tiene un" en lugar de "es un"

No toda relación entre clases es de herencia. Muchas veces un objeto **tiene** otro
objeto dentro. A eso se le llama **composición**, y modela la relación "**tiene un**".

Un coche no *es un* motor: un coche **tiene un** motor. Lo expresamos guardando una
instancia de `Motor` como atributo de `Coche`:

In [ ]:
class Motor:
    def arrancar(self):
        return "Motor en marcha"


class Coche:
    def __init__(self):
        self.motor = Motor()  # has-one: un coche TIENE un motor

    def arrancar(self):
        # el coche delega parte del trabajo en su motor
        return self.motor.arrancar()


coche = Coche()
print(coche.arrancar())

La composición también modela el "**tiene muchos**" (*has-many*): un objeto que
contiene una lista de otros. Ya lo hemos usado sin nombrarlo, por ejemplo en la
`Biblioteca` que **tiene muchos** libros:

In [ ]:
class Libro:
    def __init__(self, titulo):
        self.titulo = titulo


class Biblioteca:
    def __init__(self):
        self.libros = []  # has-many: una biblioteca TIENE muchos libros

    def añadir(self, libro):
        self.libros.append(libro)


bib = Biblioteca()
bib.añadir(Libro("1984"))
bib.añadir(Libro("Dune"))
print([libro.titulo for libro in bib.libros])

---

# Parte 3: Clases abstractas (ABC)

## 4. Definir un contrato que las subclases deben cumplir

A veces queremos definir una clase base que **no tiene sentido instanciar por sí
misma**, sino que solo sirve para marcar un **contrato**: "toda subclase mía debe
implementar estos métodos". Para eso están las **Abstract Base Classes (ABC)**.

Una `Figura`, por ejemplo, no es nada concreto (no podemos calcular "el área de una
figura genérica"), pero queremos obligar a que toda figura concreta sepa calcular su
`area`:

In [1]:
from abc import ABC, abstractmethod


class Figura(ABC):                 # hereda de ABC: es una clase abstracta
    @abstractmethod
    def area(self):                # contrato: toda figura debe implementar area()
        pass


class Circulo(Figura):
    def __init__(self, radio):
        self.radio = radio

    def area(self):
        return 3.1416 * self.radio**2


class Rectangulo(Figura):
    def __init__(self, ancho, alto):
        self.ancho = ancho
        self.alto = alto

    def area(self):
        return self.ancho * self.alto


print(Circulo(2).area())
print(Rectangulo(3, 4).area())

12.5664
12


## 5. El beneficio: fallar pronto

Lo valioso de una ABC es que Python **impide crear** objetos que no cumplan el
contrato. Si intentamos instanciar la clase abstracta directamente, o una subclase
que se ha olvidado de implementar `area`, falla **al instanciar**, no más tarde de
forma misteriosa:

In [ ]:
# 1) No se puede instanciar la propia clase abstracta
try:
    Figura()
except TypeError as e:
    print("Error:", e)


# 2) Una subclase que NO implementa area() tampoco se puede instanciar
class Triangulo(Figura):
    pass  # se nos ha olvidado definir area()


try:
    Triangulo()
except TypeError as e:
    print("Error:", e)

> 💡 "Fallar pronto" es una gran ventaja: en lugar de descubrir el problema cuando
> alguien llama a `triangulo.area()` en mitad del programa, lo detectamos justo al
> intentar crear el objeto, con un mensaje claro de qué método falta.

---

## 6. Resumen

En este vídeo hemos visto tres formas de relacionar clases y diseñar con ellas:

1. **Herencia** (*is-a*): una subclase parte de una base, hereda su comportamiento y
   puede **sobrescribir** métodos. Con `super()` ampliamos el comportamiento del
   padre en vez de reemplazarlo.
2. **Composición** (*has-a*): un objeto **tiene** otros objetos como atributos. Es
   más flexible que la herencia, y ante la duda suele ser la mejor opción.
3. **Clases abstractas (ABC)** con `@abstractmethod`: definen un **contrato** y hacen
   que el programa **falle pronto** si una subclase no lo cumple.

| Concepto | Para qué | Palabra clave |
|----------|----------|---------------|
| Herencia | Reutilizar/variar comportamiento ("es un") | `class Hija(Base)` |
| `super()` | Ampliar el método de la base sin repetirlo | `super().__init__(...)` |
| Composición | Construir un objeto a partir de otros ("tiene un") | `self.pieza = Pieza()` |
| ABC | Obligar a las subclases a implementar métodos | `ABC`, `@abstractmethod` |